# 核验 2011 年 10 月核心业务结论

使用 [02 清洗](02_clean_october_transactions.ipynb) 生成的有效明细核验客户集中度、商品发票集中和缺客户号指标。手工项使用[原始工作簿](../data_source.md#下载与校验)，以保留正负记录。对照值来自[主报告](../reports/2011年10月经营复盘.md)和[证据附录](../reports/业务证据核验附录.md)，独立于指标计算。

从 notebooks/ 启动内核，先运行 02，再执行本 Notebook。第 E 节区分已保存快照对账和需要历史产物或数据库才能复核的部分。

In [1]:
from pathlib import Path

import pandas as pd

valid_path = Path('../data/processed/data_10month.csv')
raw_path = Path('../data/raw/Online Retail.xlsx')
evidence_dir = Path('../reports/商品排名与对账')
for input_path in (valid_path, raw_path):
    if not input_path.is_file():
        raise FileNotFoundError(f'缺少 {input_path}。请参阅 ../data_source.md 和 ../README.md。')

valid = pd.read_csv(
    valid_path,
    dtype={'invoiceno': 'string', 'stockcode': 'string', 'description': 'string'},
    keep_default_na=False,
    na_values=[''],
)

def compare_report(calculated, reported):
    # 输入使用报告的展示精度。
    comparison = pd.DataFrame({
        '实际值': pd.Series(calculated),
        '报告值': pd.Series(reported),
    })
    comparison['一致'] = comparison['实际值'].eq(comparison['报告值'])
    display(comparison)
    assert comparison['一致'].all(), '存在与报告不一致的指标，请查看对照表。'

## A. 荷兰与澳大利亚的客户集中度

用有效行的 lineamount 计算金额。分子取市场内已识别客户的最大金额，分母是该市场全部有效金额。发票金额按市场内 invoiceno 汇总。偶数张取中间两张的均值作中位数。剔除均值只移除按金额排在首位的一张，同额时按发票号升序。对照主报告第 5 节。

In [2]:
market_metrics = {}
for country in ('Netherlands', 'Australia'):
    market = valid.loc[valid['country'].eq(country)]
    market_amount = market['lineamount'].sum()
    customers = market.groupby('customerid')['lineamount'].sum()
    top_customer_amount = customers.max()
    invoices = (
        market.groupby('invoiceno')['lineamount'].sum()
        .reset_index()
        .sort_values(['lineamount', 'invoiceno'], ascending=[False, True])
    )
    market_metrics[country] = {
        '市场金额（GBP）': round(market_amount, 2),
        '有效发票数': len(invoices),
        '最大客户金额（GBP）': round(top_customer_amount, 2),
        '最大客户金额占比（%）': round(100 * top_customer_amount / market_amount, 2),
        '最大一张发票金额（GBP）': round(invoices['lineamount'].iloc[0], 2),
        '平均发票金额（GBP）': round(invoices['lineamount'].mean(), 2),
        '发票金额中位数（GBP）': round(invoices['lineamount'].median(), 2),
        '剔除最大一张后的平均金额（GBP）': round(invoices['lineamount'].iloc[1:].mean(), 2),
    }

In [3]:
report_market = {
    'Netherlands': [40708.65, 15, 39995.95, 98.25, 20277.92, 2713.91, 788.16, 1459.34],
    'Australia': [17150.53, 4, 16471.77, 96.04, 15643.77, 4287.63, 708.48, 502.25],
}
for country, expected in report_market.items():
    print(country)
    compare_report(market_metrics[country], dict(zip(market_metrics[country], expected)))

Netherlands


,实际值,报告值,一致
市场金额（GBP）,40708.65,40708.65,True
有效发票数,15.00,15.00,True
最大客户金额（GBP）,39995.95,39995.95,True
最大客户金额占比（%）,98.25,98.25,True
最大一张发票金额（GBP）,20277.92,20277.92,True
平均发票金额（GBP）,2713.91,2713.91,True
发票金额中位数（GBP）,788.16,788.16,True
剔除最大一张后的平均金额（GBP）,1459.34,1459.34,True


Australia


,实际值,报告值,一致
市场金额（GBP）,17150.53,17150.53,True
有效发票数,4.00,4.00,True
最大客户金额（GBP）,16471.77,16471.77,True
最大客户金额占比（%）,96.04,96.04,True
最大一张发票金额（GBP）,15643.77,15643.77,True
平均发票金额（GBP）,4287.63,4287.63,True
发票金额中位数（GBP）,708.48,708.48,True
剔除最大一张后的平均金额（GBP）,502.25,502.25,True


## B. 商品 23084 的发票数量集中度

从有效明细筛选精确编码 23084，按发票汇总 quantity。按数量降序、发票号升序取前三张。分母是该编码在全月有效明细的数量总和。对照主报告第 7 节。

In [4]:
product_lines = valid.loc[valid['stockcode'].eq('23084')]
product_invoices = product_lines.groupby('invoiceno', as_index=False).agg(
    quantity=('quantity', 'sum'), product_amount_gbp=('lineamount', 'sum')
)
top_three = product_invoices.sort_values(
    ['quantity', 'invoiceno'], ascending=[False, True]
).head(3)
display(top_three.round({'product_amount_gbp': 2}))
top_three_quantity = top_three['quantity'].sum()
product_quantity = product_lines['quantity'].sum()
compare_report(
    {'商品总数量': product_quantity, '前三张数量': top_three_quantity,
     '前三张数量占比（%）': round(100 * top_three_quantity / product_quantity, 2)},
    {'商品总数量': 6323, '前三张数量': 4272, '前三张数量占比（%）': 67.56},
)

,invoiceno,quantity,product_amount_gbp
22,573003,2400,4992.00
14,572869,960,1718.40
34,573153,912,1632.48


,实际值,报告值,一致
商品总数量,6323.00,6323.00,True
前三张数量,4272.00,4272.00,True
前三张数量占比（%）,67.56,67.56,True


## C. 新加坡和法国的 M 手工项

在原始工作簿的 10 月窗口筛选精确编码 M。按报告列出的市场和发票号定位凭证，编号仅作定位。金额由原始带符号数量乘单价，再按市场和发票汇总。行级对照检查每张发票的数量和单价。

In [5]:
reported_m_lines = pd.DataFrame(
    [
        ('Singapore', 'C571750', -1, 3949.32), ('Singapore', 'C571750', -1, 2118.74),
        ('Singapore', '571751', 1, 3949.32), ('Singapore', '571751', 1, 2118.74),
        ('France', '573077', 1, 4161.06), ('France', 'C573079', -2, 4161.06),
        ('France', '573080', 1, 4161.06),
    ],
    columns=['Country', 'InvoiceNo', 'Quantity', 'UnitPrice'],
).astype({'InvoiceNo': 'string'})
raw = pd.read_excel(raw_path, dtype={'InvoiceNo': 'string', 'StockCode': 'string'})
raw_october = raw.loc[
    raw['InvoiceDate'].ge(pd.Timestamp('2011-10-01'))
    & raw['InvoiceDate'].lt(pd.Timestamp('2011-11-01'))
]
original_manual = raw_october.loc[raw_october['StockCode'].eq('M')]
evidence_keys = ['Country', 'InvoiceNo']
manual_lines = original_manual.merge(
    reported_m_lines[evidence_keys].drop_duplicates(),
    on=evidence_keys, validate='many_to_one',
)
manual_lines['signed_amount_gbp'] = manual_lines['Quantity'] * manual_lines['UnitPrice']
display(manual_lines.sort_values(['Country', 'InvoiceDate', 'UnitPrice'])[
    ['Country', 'InvoiceDate', 'InvoiceNo', 'CustomerID', 'Quantity',
     'UnitPrice', 'signed_amount_gbp']
].round({'signed_amount_gbp': 2}))

,Country,InvoiceDate,InvoiceNo,CustomerID,Quantity,UnitPrice,signed_amount_gbp
4,France,2011-10-27 14:13:00,573077,12536.0,1,4161.06,4161.06
5,France,2011-10-27 14:15:00,C573079,12536.0,-2,4161.06,-8322.12
6,France,2011-10-27 14:20:00,573080,12536.0,1,4161.06,4161.06
1,Singapore,2011-10-19 11:16:00,C571750,12744.0,-1,2118.74,-2118.74
0,Singapore,2011-10-19 11:16:00,C571750,12744.0,-1,3949.32,-3949.32
3,Singapore,2011-10-19 11:18:00,571751,12744.0,1,2118.74,2118.74
2,Singapore,2011-10-19 11:18:00,571751,12744.0,1,3949.32,3949.32


In [6]:
line_columns = reported_m_lines.columns.tolist()
pd.testing.assert_frame_equal(
    manual_lines[line_columns].sort_values(line_columns).reset_index(drop=True),
    reported_m_lines.sort_values(line_columns).reset_index(drop=True),
)
print(f'原始行的市场、发票、数量及单价全部与证据附录一致，共 {len(manual_lines)} 行。')

signed_totals = manual_lines.groupby(evidence_keys)['signed_amount_gbp'].sum().round(2)
compare_report(signed_totals.to_dict(), {
    ('Singapore', 'C571750'): -6068.06, ('Singapore', '571751'): 6068.06,
    ('France', '573077'): 4161.06, ('France', 'C573079'): -8322.12,
    ('France', '573080'): 4161.06,
})
positive_evidence = manual_lines.loc[
    manual_lines['signed_amount_gbp'].gt(0), 'signed_amount_gbp'
].sum()
valid_manual_amount = valid.loc[valid['stockcode'].eq('M'), 'lineamount'].sum()
compare_report(
    {'指定正向票金额（GBP）': round(positive_evidence, 2),
     '该编码正向总额（GBP）': round(valid_manual_amount, 2),
     '指定正向票金额占比（%）': round(100 * positive_evidence / valid_manual_amount, 2)},
    {'指定正向票金额（GBP）': 14390.18, '该编码正向总额（GBP）': 21116.67,
     '指定正向票金额占比（%）': 68.15},
)

原始行的市场、发票、数量及单价全部与证据附录一致，共 7 行。


实际值      报告值    一致
France    573077   4161.06  4161.06  True
          573080   4161.06  4161.06  True
          C573079 -8322.12 -8322.12  True
Singapore 571751   6068.06  6068.06  True
          C571750 -6068.06 -6068.06  True

,实际值,报告值,一致
指定正向票金额（GBP）,14390.18,14390.18,True
该编码正向总额（GBP）,21116.67,21116.67,True
指定正向票金额占比（%）,68.15,68.15,True


新加坡两张凭证在两个单价上分别出现数量等值反向的记录。法国三张凭证单价相同，数量依时间为 +1、−2、+1。时间、符号和金额提供了关联线索。确认是否退款、退货或重开需要原始凭证与业务说明。

## D. 缺失 CustomerID 的影响

用仍保留缺失客户号的有效明细。金额分子只计缺失客户号的行，金额分母为全月正向金额。发票分子按至少一行缺客户号的发票号去重，分母为全月有效发票号去重数。行数占比以有效明细总行数为分母。对照主报告第 5 节。

In [7]:
no_customer = valid.loc[valid['customerid'].isna()]
no_customer_amount = no_customer['lineamount'].sum()
affected_invoices = no_customer['invoiceno'].nunique()
compare_report(
    {'缺客户号明细行数': len(no_customer),
     '缺客户号明细行占比（%）': round(100 * len(no_customer) / len(valid), 2),
     '缺客户号明细金额（GBP）': round(no_customer_amount, 2),
     '缺客户号明细金额占比（%）': round(100 * no_customer_amount / valid['lineamount'].sum(), 2),
     '至少一行缺客户号的发票数': affected_invoices,
     '至少一行缺客户号的发票占比（%）': round(
         100 * affected_invoices / valid['invoiceno'].nunique(), 2
     )},
    {'缺客户号明细行数': 9738, '缺客户号明细行占比（%）': 16.84,
     '缺客户号明细金额（GBP）': 115582.05, '缺客户号明细金额占比（%）': 10.07,
     '至少一行缺客户号的发票数': 111, '至少一行缺客户号的发票占比（%）': 5.44},
)

,实际值,报告值,一致
缺客户号明细行数,9738.00,9738.00,True
缺客户号明细行占比（%）,16.84,16.84,True
缺客户号明细金额（GBP）,115582.05,115582.05,True
缺客户号明细金额占比（%）,10.07,10.07,True
至少一行缺客户号的发票数,111.00,111.00,True
至少一行缺客户号的发票占比（%）,5.44,5.44,True


## E. 对照可以取得的历史快照

从有效明细重新生成冲突候选频次，与已发布的候选 CSV 比较。原先重构前的完整中间表没有随项目保存，因此原先逐项相等断言**当前无法独立复现**。

商品部分从有效明细独立聚合，按[商品 SQL](../sql/02_product_statistics_and_rankings.sql)相同的排除清单与排名规则核对已保存的两个 Top 10。项目没有保存历史核对所用的完整 MySQL 导出。该历史全量对账**当前无法独立复现**，此处比较可取得的榜单快照。

In [8]:
normalized = valid['description'].str.strip().str.lower()
description_rows = valid.assign(normalized_candidate_description=normalized)
description_rows = description_rows.loc[normalized.notna() & normalized.ne('')]
candidate_columns = ['stockcode', 'normalized_candidate_description']
candidate_counts = description_rows.groupby(candidate_columns).size().reset_index(
    name='detail_row_count'
)
variant_count = candidate_counts.groupby('stockcode')['stockcode'].transform('size')
conflicts = candidate_counts.loc[variant_count.gt(1)]
saved_conflicts = pd.read_csv(
    evidence_dir / '冲突描述候选及频次.csv',
    dtype={'stockcode': 'string', 'normalized_candidate_description': 'string'},
)
count_columns = [*candidate_columns, 'detail_row_count']
pd.testing.assert_frame_equal(
    conflicts[count_columns].sort_values(candidate_columns).reset_index(drop=True),
    saved_conflicts[count_columns].sort_values(candidate_columns).reset_index(drop=True),
)
print(f'描述候选与发布快照一致：{len(conflicts)} 条、{conflicts["stockcode"].nunique()} 个编码。')

描述候选与发布快照一致：128 条、62 个编码。


In [9]:
product_facts = valid.groupby('stockcode', as_index=False).agg(
    gross_amount_gbp=('lineamount', 'sum'),
    units=('quantity', 'sum'),
    invoice_coverage=('invoiceno', 'nunique'),
)
product_facts['gross_amount_gbp'] = product_facts['gross_amount_gbp'].round(2)
product_facts['stockcode_key'] = product_facts['stockcode'].map(
    lambda code: code.encode('utf-8').hex().upper()
)
excluded = {'M', 'DOT', 'POST', 'C2', 'BANK CHARGES', 'S', 'gift_0001_20'}
regular = product_facts.loc[~product_facts['stockcode'].isin(excluded)]
fields = ['stockcode_key', 'stockcode', 'gross_amount_gbp', 'units', 'invoice_coverage']
for metric, filename in [
    ('gross_amount_gbp', '普通商品_金额Top10.csv'),
    ('invoice_coverage', '普通商品_发票覆盖Top10.csv'),
]:
    calculated = regular.sort_values([metric, 'stockcode_key'], ascending=[False, True]).head(10)
    saved = pd.read_csv(evidence_dir / filename, dtype={'stockcode': 'string', 'stockcode_key': str})
    pd.testing.assert_frame_equal(
        calculated[fields].reset_index(drop=True), saved[fields].reset_index(drop=True)
    )
    print(f'{filename}：前 {len(calculated)} 名的顺序、金额、数量与覆盖均一致。')

普通商品_金额Top10.csv：前 10 名的顺序、金额、数量与覆盖均一致。
普通商品_发票覆盖Top10.csv：前 10 名的顺序、金额、数量与覆盖均一致。
